Aim

To implement Triplet Loss using TensorFlow and Keras for learning similarity between data samples.

In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Flatten,
    Dropout,
    Lambda
)

from tensorflow.keras.optimizers import RMSprop

# Input Configuration
image_dimension = (28, 28)
feature_size = 32

# Create Feature Extraction Network
def build_feature_model():

    input_layer = Input(shape=image_dimension)

    flatten_layer = Flatten()(input_layer)

    hidden_layer1 = Dense(
        256,
        activation='relu'
    )(flatten_layer)

    dropout_layer = Dropout(0.3)(
        hidden_layer1
    )

    hidden_layer2 = Dense(
        128,
        activation='relu'
    )(dropout_layer)

    output_layer = Dense(
        feature_size,
        activation='linear'
    )(hidden_layer2)

    return Model(
        input_layer,
        output_layer
    )

# Create Shared Neural Network
feature_network = build_feature_model()

# Input Layers
anchor_image = Input(shape=image_dimension)
similar_image = Input(shape=image_dimension)
different_image = Input(shape=image_dimension)

# Feature Vectors
anchor_vector = feature_network(anchor_image)

similar_vector = feature_network(similar_image)

different_vector = feature_network(different_image)

# Distance Calculation Function
def calculate_distance(inputs):

    first_vector, second_vector = inputs

    squared_difference = tf.square(
        first_vector - second_vector
    )

    return tf.reduce_sum(
        squared_difference,
        axis=1,
        keepdims=True
    )

# Positive Distance
positive_score = Lambda(
    calculate_distance
)([
    anchor_vector,
    similar_vector
])

# Negative Distance
negative_score = Lambda(
    calculate_distance
)([
    anchor_vector,
    different_vector
])

# Triplet Difference
triplet_difference = Lambda(
    lambda values: values[0] - values[1]
)([
    positive_score,
    negative_score
])

# Create Final Model
deep_triplet_model = Model(
    inputs=[
        anchor_image,
        similar_image,
        different_image
    ],
    outputs=triplet_difference
)

# Triplet Loss Function
def custom_triplet_loss(y_true, y_pred):

    alpha = 0.5

    loss_value = tf.maximum(
        y_pred + alpha,
        0.0
    )

    return tf.reduce_mean(loss_value)

# Compile Model
deep_triplet_model.compile(
    optimizer=RMSprop(),
    loss=custom_triplet_loss
)

# Generate Random Dataset
sample_count = 120

anchor_samples = np.random.random(
    (sample_count, 28, 28)
)

positive_samples = np.random.random(
    (sample_count, 28, 28)
)

negative_samples = np.random.random(
    (sample_count, 28, 28)
)

labels = np.zeros((sample_count, 1))

# Train Model
training_output = deep_triplet_model.fit(
    [
        anchor_samples,
        positive_samples,
        negative_samples
    ],
    labels,
    epochs=12,
    batch_size=12,
    verbose=1
)

print("\nDeep Triplet Loss Model Training Completed")

Epoch 1/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 4.0955
Epoch 2/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 2.1015
Epoch 3/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 1.1309
Epoch 4/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.9198
Epoch 5/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.7459
Epoch 6/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.5445
Epoch 7/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.5395
Epoch 8/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.4816
Epoch 9/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.4819
Epoch 10/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.4251
Epoch 11/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.5086
Epoch 12/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.3121

Deep Triplet Loss Model Training Completed


Conclusion

Successfully implemented Triplet Loss using TensorFlow and Keras for learning embeddings and similarity representation between samples.